# 第 10 章练习

完成下面的练习，结合自己的运行结果作答。

分析本次耗时前，先阅读实验 Notebook 中“怎样理解本次 Device 比 CPU 慢”，按应用总时间和线性求解时间分别比较。


<a id="s-10-06_chapter_test"></a>


## 10.6 章节实践

下面提供运行和读取结果的单元格。可以复用上一实验的结果；需要换矩阵或重测时再运行。完成末尾的一组练习即可。


### 检查设备和 CMake

下面显示 CANN 目录、CMake 版本和 NPU 信息。


In [ ]:
%%bash
set -e
echo "CANN 目录：$ASCEND_HOME_PATH"
cmake --version
npu-smi info


### 运行实验

下面设置结果目录。需要保留两次结果进行比较时，使用不同名称。


In [ ]:
# 学生修改：本次结果目录。
%env RESULT_DIR=results/chapter_test_01


In [ ]:
%%bash
set -euo pipefail
cd src/ascend_xyce
cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DASCEND_XYCE_HOST_ONLY=OFF
cmake --build build -j
mkdir -p results
mkdir -p "$RESULT_DIR"
# 学生修改：--matrix 选择输入矩阵；本题先使用 U1。
./build/bin/xyce_benchmark --warmup 0 --repeat 1 \
  --matrix U1 --csv "$RESULT_DIR/result.csv" 2>&1 | tee "$RESULT_DIR/run.log"


### 读取结果

程序已检查收敛和误差。下面列出三种求解器的结果，比较它们的迭代数与耗时。

复用上一实验结果时，跳过“运行实验”，把下面的 `result_file` 一行改为 `result_file = Path("src/ascend_xyce/results/course_u1.csv")`。本节重新运行后，保持默认代码。


In [ ]:
import csv
import os
from pathlib import Path

result_file = Path("src/ascend_xyce") / os.environ["RESULT_DIR"] / "result.csv"
with result_file.open() as stream:
    for row in csv.DictReader(stream):
        print(row["solver"], "残差：", row["final_residual"],
              "解误差：", row["solution_error_vs_expected"],
              "求解耗时：", row["linear_solver_ms"], "ms")


### 练习

基础题对照源码作答；运行题使用前面已经生成的结果，无需再跑一遍。

#### 1. 基础题

（1）`xyce_adapter.hpp` 中定义了哪三种求解器？（单选）

A. `CpuSingle`、`CpuOpenMP16`、`AscendDevice`

B. `CpuSingle`、`CpuOpenMP16`、`AscendAccelerated`

C. `CpuSingle`、`CpuOpenMP16`、`AscendDevice`、`HcclDistributed`

D. `CpuSingle`、`AscendAccelerated`

（2）判断：设备求解器使用第 09 章的 `dis_gmres` 源码；本章附带的 `backend/gmres/Ascend-GMRES` 提供 CPU 实现和旧 Host 原型。请在 CMake 和适配器调用处找到依据。

（3）`xyce_benchmark` 输出的 CSV 结构是怎样的？（单选）

A. 没有表头，每个矩阵对应三行求解结果

B. 整个文件一行表头，每个矩阵对应三行求解结果

C. 整个文件一行表头，只有一行求解结果

D. 不论选择几个矩阵，整个文件都只有一行表头和三行结果

（4）判断：本章运行的是线性求解器测试程序，用来比较三种求解器的结果和耗时；完成一次完整电路仿真，还需要 Xyce 读取电路网表并执行仿真。

#### 2. 运行与校验

使用本节或上一实验的 U1 日志和 CSV。找出 CPU 与设备求解器分别使用的源码位置，检查三行结果中的收敛状态、残差和解误差。

#### 3. 解释耗时和错误

查看 CSV 中矩阵装配、线性求解、SpMV 和总时间的字段，说明它们分别包含哪些工作、哪些时间有包含关系。

再用未知参数或不可写的结果目录做一次失败实验，记录命令、退出码和错误输出，判断问题出在参数、构建、设备运行还是结果写入。

#### 4. 总结结果

用表格列出三种求解器的迭代数、残差、解误差和耗时，结合源码解释它们分别在哪里计算。


[参考答案](answer/10.06_answer.md)
